In [1]:
#导入库
from transformers import AutoModelForCausalLM, AutoTokenizer

ModuleNotFoundError: No module named 'transformers'

In [25]:
#从预训练的模型里面加载参数
model = AutoModelForCausalLM.from_pretrained(
    'Qwen2.5-0.5B-Instruct',
    torch_dtype='auto'
)
#print(model)

In [26]:
#加载token模型
tokenizer = AutoTokenizer.from_pretrained('Qwen2.5-0.5B-Instruct')

In [27]:
#给大模型的输入;system时全局设定;user给大模型的问题
message = [
    {'role': 'system', 'content': '你是千问，一个有用的人工智能。'},
    {'role': 'user', 'content': '我爱北京'}
]

In [28]:
#给上面的message加标记;最后一行<|im_start|>assistant告诉大模型可以回答问题了
text = tokenizer.apply_chat_template(
    message,
    tokenize=False,
    add_generation_prompt=True
)
#print(text)

In [29]:
#把text变成token,即数字;attention_mask全是1表示大模型可以看到全部的输入
model_inputs = tokenizer([text], return_tensors='pt').to('cpu')
print(model_inputs)

{'input_ids': tensor([[151644,   8948,    198, 105043,  99320,  56007,   3837,  46944, 115404,
         100623,  48692, 100168,   1773, 151645,    198, 151644,    872,    198,
          35946,  99242,  68990, 151645,    198, 151644,  77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1]])}


In [30]:
#大模型生成回答
generated_ids = model.generate(**model_inputs, max_new_tokens=512)
generated_ids

tensor([[151644,   8948,    198, 105043,  99320,  56007,   3837,  46944, 115404,
         100623,  48692, 100168,   1773, 151645,    198, 151644,    872,    198,
          35946,  99242,  68990, 151645,    198, 151644,  77091,    198,  99491,
         102483, 102804,  47874,   6313, 106870, 110117, 101888,  68990, 103936,
          57191,  85106, 100364,  37945, 102422, 106525,   1773, 151645]])

In [33]:
#把输入部分去掉
generated_ids = [output_ids[len(input_ids):]
                            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
generated_ids 

[tensor([ 99491, 102483, 102804,  47874,   6313, 106870, 110117, 101888,  68990,
         103936,  57191,  85106, 100364,  37945, 102422, 106525,   1773, 151645])]

In [34]:
#把数字转换成文字
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [35]:
response

'非常高兴为您服务！如果您有任何关于北京的问题或需要帮助，请随时告诉我。'

In [36]:
!pip install datasets trl peft

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple, https://pypi.mirrors.ustc.edu.cn/simple


In [37]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

dataset = load_dataset("tatsu-lab-alpaca", split="train")

peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [38]:
trainer = SFTTrainer(
    'Qwen2.5-0.5B-Instruct',
    train_dataset=dataset,
    args=SFTConfig(output_dir="tmp"),
    peft_config=peft_config,
    dataset_text_field='instruction'
)

C:\Users\86135\anaconda3\envs\llms\Lib\site-packages\huggingface_hub\utils\_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
C:\Users\86135\anaconda3\envs\llms\Lib\site-packages\trl\trainer\sft_trainer.py:202: UserWarning: You passed a model_id to the SFTTrainer. This will automatically create an `AutoModelForCausalLM` or a `PeftModel` (if you passed a `peft_config`) for you.
  warnings.warn(
C:\Users\86135\anaconda3\envs\llms\Lib\site-packages\trl\trainer\sft_trainer.py:309: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(
C:\Users\86135\anaconda3\envs\llms\Lib\site-packages\trl\trainer\sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` argument to the SFTTra

In [ ]:
trainer.train()

Step,Training Loss
